# 从零实现 Speculative Decoding：提议、验证、拒绝残差与 cache rollback

本 Notebook 用手写 BigramLM57 与 SpeculativeDecoder57 展示 speculative sampling 的完整概率语义：draft 顺序提议，target 以 min(1,p/q) 验证，拒绝时从归一化正部 (p-q)+ 采样替代 token，全接受时再由 target 产生 bonus token。

实现显式处理 EOS、torch.Generator、请求级 cache 位置与拒绝回滚。它不用 transformers、vLLM 或现成 speculative decoder；小词表 bigram 只是精确概率 oracle，不代表真实 LLM 的吞吐收益。

In [ ]:
import copy
import hashlib
import json
import math
import random
import warnings
from types import MappingProxyType

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")
import numpy as np
import torch
from torch import nn

SEED57 = 5701
random.seed(SEED57); np.random.seed(SEED57); torch.manual_seed(SEED57)
torch.set_num_threads(1)
DEVICE57 = torch.device("cpu")

def canonical_json57(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def sha57(raw):
    return hashlib.sha256(raw).hexdigest()

assert DEVICE57.type == "cpu"
assert torch.get_num_threads() == 1

## 1. Token、评估 prompt 与概率矩阵

词表只有 BOS、A、B、EOS。target/draft 都是一阶因果模型，transition[current,next] 每行和为 1。draft 故意在 BOS 后过度偏好 B，让拒绝路径可观测；target/draft tokenizer 必须完全相同。

PROMPTS57 与 SPLIT57 会被完整写入发布 metadata。calibration 可用于选择 gamma，test 只做机制回归；这里没有声称对真实请求分布有效。

In [ ]:
TOKENIZER57 = {"<bos>": 0, "A": 1, "B": 2, "<eos>": 3}
PROMPTS57 = [
    {"id": "cal-bos", "tokens": [0]},
    {"id": "cal-a", "tokens": [0, 1]},
    {"id": "test-b", "tokens": [0, 2]},
]
SPLIT57 = {"calibration": ["cal-bos", "cal-a"], "test": ["test-b"]}
TARGET_PROBS57 = torch.tensor([
    [0.02, 0.52, 0.38, 0.08],
    [0.02, 0.10, 0.30, 0.58],
    [0.02, 0.25, 0.15, 0.58],
    [0.01, 0.01, 0.01, 0.97],
], dtype=torch.float64)
DRAFT_PROBS57 = torch.tensor([
    [0.02, 0.25, 0.65, 0.08],
    [0.02, 0.15, 0.20, 0.63],
    [0.02, 0.45, 0.08, 0.45],
    [0.01, 0.01, 0.01, 0.97],
], dtype=torch.float64)
assert torch.allclose(TARGET_PROBS57.sum(-1), torch.ones(4, dtype=torch.float64))
assert torch.allclose(DRAFT_PROBS57.sum(-1), torch.ones(4, dtype=torch.float64))
assert (TARGET_PROBS57 >= 0).all() and (DRAFT_PROBS57 >= 0).all()
assert set(sum(SPLIT57.values(), [])) == {row["id"] for row in PROMPTS57}
assert not (set(SPLIT57["calibration"]) & set(SPLIT57["test"]))

## 2. 手写 causal bigram nn.Module

TransitionHead57 保存 logits:[V,V]；给定 last_tokens:[B] 返回 next_logits:[B,V]。BigramLM57 对 input_ids:[B,T] 的每个位置独立查表，因此 logits[:,t] 只条件于当前位置 token，不偷看未来。

真实 LLM 会用多层 attention 和 KV cache；选择 bigram 是为了能精确穷举 p/q，而不是逃避自回归接口、请求位置与缓存语义。

In [ ]:
class TransitionHead57(nn.Module):
    def __init__(self, probabilities):
        super().__init__()
        if probabilities.ndim != 2 or probabilities.shape[0] != probabilities.shape[1]:
            raise ValueError("transition_must_be_square")
        if not torch.isfinite(probabilities).all() or (probabilities < 0).any():
            raise ValueError("invalid_transition_probabilities")
        if not torch.allclose(probabilities.sum(-1), torch.ones(probabilities.shape[0], dtype=probabilities.dtype), atol=1e-8):
            raise ValueError("transition_rows_must_sum_to_one")
        logits = probabilities.clamp_min(1e-30).log().float()
        self.logits = nn.Parameter(logits, requires_grad=False)
        self.vocab_size = probabilities.shape[0]

    def forward(self, last_tokens):
        if last_tokens.dtype != torch.long or last_tokens.ndim < 1:
            raise ValueError("last_tokens_must_be_long")
        if last_tokens.min() < 0 or last_tokens.max() >= self.vocab_size:
            raise ValueError("token_id_out_of_range")
        return self.logits[last_tokens]

class BigramLM57(nn.Module):
    def __init__(self, probabilities):
        super().__init__()
        self.transition = TransitionHead57(probabilities)
        self.vocab_size = self.transition.vocab_size

    def forward(self, input_ids):
        if input_ids.ndim != 2 or input_ids.shape[1] < 1:
            raise ValueError("input_ids_must_have_shape_B_T")
        return self.transition(input_ids)

    @torch.no_grad()
    def next_probs(self, token):
        token_tensor = torch.tensor([token], dtype=torch.long)
        return self.transition(token_tensor)[0].softmax(-1).double()

target_model57 = BigramLM57(TARGET_PROBS57)
draft_model57 = BigramLM57(DRAFT_PROBS57)
model_probe57 = target_model57(torch.tensor([[0, 1, 2]]))
assert model_probe57.shape == (1, 3, 4)
assert torch.allclose(target_model57.next_probs(0), TARGET_PROBS57[0], atol=1e-7)
assert not any(p.requires_grad for p in target_model57.parameters())

## 3. 为什么接受概率与拒绝残差能恢复 target

draft 先以 q(x) 提议 x，接受概率 a(x)=min(1,p(x)/q(x))。被接受的质量为 q(x)a(x)=min(p(x),q(x))。总拒绝质量 Z=sum(q-p)+，而归一化残差 r(x)=(p(x)-q(x))+/Z。

因为 p、q 都归一化，sum(q-p)+=sum(p-q)+=Z，所以最终首个输出质量为 min(p,q)+Zr=p。下面对词表中每个上下文精确计算，不依赖蒙特卡洛碰巧接近。

In [ ]:
def validate_distribution57(probabilities, name):
    if probabilities.ndim != 1 or not torch.isfinite(probabilities).all() or (probabilities < 0).any():
        raise ValueError(name + "_invalid")
    if not torch.allclose(probabilities.sum(), torch.tensor(1.0, dtype=probabilities.dtype), atol=1e-8):
        raise ValueError(name + "_not_normalized")

def residual_distribution57(p, q):
    validate_distribution57(p, "target"); validate_distribution57(q, "draft")
    positive = (p - q).clamp_min(0)
    mass = positive.sum()
    if mass <= 1e-15:
        raise RuntimeError("residual_requested_when_p_equals_q")
    return positive / mass

def exact_first_output57(p, q):
    validate_distribution57(p, "target"); validate_distribution57(q, "draft")
    accepted_mass = torch.minimum(p, q)
    reject_mass = (q - p).clamp_min(0).sum()
    if reject_mass <= 1e-15:
        return accepted_mass
    return accepted_mass + reject_mass * residual_distribution57(p, q)

for context57 in range(len(TOKENIZER57)):
    exact57 = exact_first_output57(TARGET_PROBS57[context57], DRAFT_PROBS57[context57])
    assert torch.allclose(exact57, TARGET_PROBS57[context57], atol=1e-12)
    assert torch.allclose(exact57.sum(), torch.tensor(1.0, dtype=torch.float64))

p_zero_case57 = torch.tensor([0.70, 0.30, 0.0], dtype=torch.float64)
q_zero_case57 = torch.tensor([0.0, 1.0, 0.0], dtype=torch.float64)
assert torch.allclose(exact_first_output57(p_zero_case57, q_zero_case57), p_zero_case57, atol=1e-12)
assert residual_distribution57(p_zero_case57, q_zero_case57).tolist() == [1.0, 0.0, 0.0]

## 4. 单 token 验证、q=0 与全接受/全拒绝

实际 proposal 必须来自 q；若某 token 的 q=0 却声称 draft 提议了它，说明采样器或数值管线违约，应 fail closed，而不是静默除零。全接受时原样输出 proposals；第一次拒绝后只输出已接受前缀和一个 residual replacement，并结束当前 block。

In [ ]:
def sample_categorical57(probabilities, generator):
    validate_distribution57(probabilities, "sampling")
    if not isinstance(generator, torch.Generator):
        raise TypeError("explicit_torch_generator_required")
    return int(torch.multinomial(probabilities.float(), 1, generator=generator).item())

def acceptance_probability57(p, q, proposal):
    validate_distribution57(p, "target"); validate_distribution57(q, "draft")
    if not isinstance(proposal, int) or not 0 <= proposal < len(p):
        raise ValueError("proposal_out_of_range")
    if q[proposal] <= 0:
        raise ValueError("draft_proposed_zero_probability_token")
    return min(1.0, float(p[proposal] / q[proposal]))

def verify_block57(proposals, p_rows, q_rows, uniforms, generator):
    if not (len(proposals) == len(p_rows) == len(q_rows) == len(uniforms)):
        raise ValueError("verify_block_lengths_mismatch")
    emitted, accepted = [], 0
    for proposal, p, q, uniform in zip(proposals, p_rows, q_rows, uniforms):
        if not 0 <= uniform < 1:
            raise ValueError("uniform_out_of_range")
        if uniform < acceptance_probability57(p, q, proposal):
            emitted.append(proposal); accepted += 1
        else:
            replacement = sample_categorical57(residual_distribution57(p, q), generator)
            emitted.append(replacement)
            return emitted, accepted, True
    return emitted, accepted, False

equal_dist57 = torch.tensor([0.2, 0.8], dtype=torch.float64)
all_accept57 = verify_block57([1, 0], [equal_dist57, equal_dist57], [equal_dist57, equal_dist57], [0.99, 0.2], torch.Generator().manual_seed(1))
assert all_accept57 == ([1, 0], 2, False)
reject_p57 = torch.tensor([1.0, 0.0], dtype=torch.float64)
reject_q57 = torch.tensor([0.0, 1.0], dtype=torch.float64)
all_reject57 = verify_block57([1], [reject_p57], [reject_q57], [0.5], torch.Generator().manual_seed(2))
assert all_reject57 == ([0], 0, True)
assert acceptance_probability57(reject_p57, reject_q57, 1) == 0.0
boundary_p57 = torch.tensor([1.0, 0.0], dtype=torch.float64)
boundary_q57 = torch.tensor([0.5, 0.5], dtype=torch.float64)
boundary_reject57 = verify_block57([1], [boundary_p57], [boundary_q57], [0.0], torch.Generator().manual_seed(3))
assert boundary_reject57 == ([0], 0, True)
ratio_p57 = torch.tensor([0.60, 0.40], dtype=torch.float64)
ratio_q57 = torch.tensor([0.30, 0.70], dtype=torch.float64)
assert acceptance_probability57(ratio_p57, ratio_q57, 0) == 1.0
assert abs(acceptance_probability57(ratio_p57, ratio_q57, 1) - 4 / 7) < 1e-12
try:
    acceptance_probability57(reject_p57, reject_q57, 0)
    raise AssertionError("q=0 proposal was accepted")
except ValueError as error57:
    assert str(error57) == "draft_proposed_zero_probability_token"

## 5. 请求级 cache 与 rollback

TokenCache57 用 request_id 隔离状态。position 定义为当前已提交 token 数；draft 可在一个 block 内超前，target 拒绝后必须把 draft 回滚到“已接受前缀”，再把 residual replacement 同时追加到两边。

真实 KV cache 回滚的是每层 K/V 页和有效长度，而不是 token list；但位置不变量相同：一个请求的 rollback 不能影响另一个请求，也不能让 target/draft 位置分叉。

In [ ]:
class TokenCache57:
    def __init__(self):
        self._requests = {}

    def begin(self, request_id, prefix):
        if not isinstance(request_id, str) or not request_id:
            raise ValueError("request_id_must_be_nonempty_string")
        if request_id in self._requests:
            raise RuntimeError("request_already_exists")
        if not prefix:
            raise ValueError("prefix_must_be_nonempty")
        self._requests[request_id] = list(prefix)

    def append(self, request_id, token):
        if request_id not in self._requests:
            raise KeyError("unknown_request")
        self._requests[request_id].append(int(token))

    def rollback(self, request_id, position):
        if request_id not in self._requests:
            raise KeyError("unknown_request")
        if not 1 <= position <= len(self._requests[request_id]):
            raise ValueError("invalid_rollback_position")
        removed = len(self._requests[request_id]) - position
        del self._requests[request_id][position:]
        return removed

    def tokens(self, request_id):
        if request_id not in self._requests:
            raise KeyError("unknown_request")
        return list(self._requests[request_id])

    def position(self, request_id):
        return len(self.tokens(request_id))

    def release(self, request_id):
        if request_id not in self._requests:
            raise KeyError("unknown_request")
        del self._requests[request_id]

cache_probe57 = TokenCache57()
cache_probe57.begin("r-a", [0]); cache_probe57.begin("r-b", [0, 1])
cache_probe57.append("r-a", 2); cache_probe57.append("r-a", 2)
removed57 = cache_probe57.rollback("r-a", 2)
assert removed57 == 1 and cache_probe57.tokens("r-a") == [0, 2]
assert cache_probe57.tokens("r-b") == [0, 1]
assert cache_probe57.position("r-a") == cache_probe57.position("r-b") == 2

## 6. Speculative propose → verify → residual/bonus

gamma 是每个 block 的最大 proposal 数。draft 串行产生候选；target 在教学代码里逐个算 p 以便阅读。若 proposal 被接受，就提交到 target cache；拒绝则从 residual 采样 replacement、回滚 draft 并结束 block。全部接受且预算/EOS 允许时，target 再产生一个 bonus token。

输出长度严格不超过 max_new_tokens。EOS 一经提交立即停止，EOS proposal 被拒绝时则按 replacement 继续，这两个分支不能混淆。

In [ ]:
class SpeculativeDecoder57(nn.Module):
    def __init__(self, target, draft, gamma=3, eos_token=3):
        super().__init__()
        if target.vocab_size != draft.vocab_size or gamma < 1:
            raise ValueError("incompatible_models_or_gamma")
        if not 0 <= eos_token < target.vocab_size:
            raise ValueError("invalid_eos_token")
        self.target, self.draft = target, draft
        self.gamma, self.eos_token = int(gamma), int(eos_token)
        self.target_cache, self.draft_cache = TokenCache57(), TokenCache57()

    @torch.no_grad()
    def forward(self, prefix, max_new_tokens, generator, request_id):
        if not isinstance(prefix, list) or not prefix or any(type(token) is not int or not 0 <= token < self.target.vocab_size for token in prefix):
            raise ValueError("invalid_prefix")
        if prefix[-1] == self.eos_token:
            raise ValueError("prefix_already_ended")
        if not isinstance(max_new_tokens, int) or not 1 <= max_new_tokens <= 64:
            raise ValueError("invalid_max_new_tokens")
        if not isinstance(generator, torch.Generator):
            raise TypeError("explicit_torch_generator_required")
        self.target_cache.begin(request_id, prefix)
        self.draft_cache.begin(request_id, prefix)
        prefix_length = len(prefix)
        trace = {"blocks": 0, "proposed": 0, "accepted": 0, "rejected": 0, "bonuses": 0, "rollbacks": 0}

        while self.target_cache.position(request_id) - prefix_length < max_new_tokens:
            trace["blocks"] += 1
            remaining = max_new_tokens - (self.target_cache.position(request_id) - prefix_length)
            proposal_budget = min(self.gamma, remaining)
            proposals, q_rows = [], []
            for _ in range(proposal_budget):
                current = self.draft_cache.tokens(request_id)[-1]
                q = self.draft.next_probs(current)
                proposal = sample_categorical57(q, generator)
                proposals.append(proposal); q_rows.append(q)
                self.draft_cache.append(request_id, proposal)
                trace["proposed"] += 1
                if proposal == self.eos_token:
                    break

            rejected = False
            for proposal, q in zip(proposals, q_rows):
                current = self.target_cache.tokens(request_id)[-1]
                p = self.target.next_probs(current)
                uniform = float(torch.rand((), generator=generator))
                if uniform < acceptance_probability57(p, q, proposal):
                    self.target_cache.append(request_id, proposal)
                    trace["accepted"] += 1
                    if proposal == self.eos_token:
                        break
                else:
                    replacement = sample_categorical57(residual_distribution57(p, q), generator)
                    self.target_cache.append(request_id, replacement)
                    keep_before_replacement = self.target_cache.position(request_id) - 1
                    trace["rollbacks"] += self.draft_cache.rollback(request_id, keep_before_replacement)
                    self.draft_cache.append(request_id, replacement)
                    trace["rejected"] += 1
                    rejected = True
                    break

            generated = self.target_cache.tokens(request_id)[prefix_length:]
            ended = bool(generated and generated[-1] == self.eos_token)
            if not rejected and not ended and len(generated) < max_new_tokens:
                p_bonus = self.target.next_probs(self.target_cache.tokens(request_id)[-1])
                bonus = sample_categorical57(p_bonus, generator)
                self.target_cache.append(request_id, bonus)
                self.draft_cache.append(request_id, bonus)
                trace["bonuses"] += 1
                generated.append(bonus)
                ended = bonus == self.eos_token
            if ended:
                break

        if self.target_cache.tokens(request_id) != self.draft_cache.tokens(request_id):
            raise RuntimeError("target_draft_cache_diverged")
        return self.target_cache.tokens(request_id)[prefix_length:prefix_length + max_new_tokens], trace

    def forward_batch(self, requests, max_new_tokens, generators):
        if len(requests) != len(generators):
            raise ValueError("batch_generator_count_mismatch")
        outputs = {}
        for request, generator in zip(requests, generators):
            request_id = request["request_id"]
            outputs[request_id] = self(request["prefix"], max_new_tokens, generator, request_id)
        return outputs

    def release_request(self, request_id):
        self.target_cache.release(request_id)
        self.draft_cache.release(request_id)

decoder57 = SpeculativeDecoder57(target_model57, draft_model57, gamma=2, eos_token=TOKENIZER57["<eos>"])
generated57, trace57 = decoder57([0], 5, torch.Generator().manual_seed(71), "main")
assert 1 <= len(generated57) <= 5
assert trace57["proposed"] >= trace57["accepted"]
assert trace57["rejected"] <= trace57["blocks"]
assert decoder57.target_cache.tokens("main") == decoder57.draft_cache.tokens("main")
assert decoder57.target_cache.position("main") == 1 + len(generated57)
decoder57.release_request("main")

## 7. 全接受、全拒绝、bonus 与 EOS oracle

p=q 时每个 proposal 接受率为 1；若预算仍有空间，target bonus 必须出现。分布近乎不相交时，第一个 proposal 应拒绝，draft 超前 token 被 rollback，replacement 来自 p-q 正部。target/draft 都确定性指向 EOS 时，只生成一个 EOS 且没有 bonus。

In [ ]:
def deterministic_matrix57(next_token):
    matrix = torch.zeros(4, 4, dtype=torch.float64)
    matrix[:, next_token] = 1.0
    return matrix

cycle57 = deterministic_matrix57(1)
accept_decoder57 = SpeculativeDecoder57(BigramLM57(cycle57), BigramLM57(cycle57), gamma=2)
accept_tokens57, accept_trace57 = accept_decoder57([0], 3, torch.Generator().manual_seed(10), "accept")
assert accept_tokens57 == [1, 1, 1]
assert accept_trace57["accepted"] == 2
assert accept_trace57["rejected"] == 0 and accept_trace57["bonuses"] == 1
assert accept_trace57["rollbacks"] == 0

reject_decoder57 = SpeculativeDecoder57(BigramLM57(deterministic_matrix57(1)), BigramLM57(deterministic_matrix57(2)), gamma=3)
reject_tokens57, reject_trace57 = reject_decoder57([0], 1, torch.Generator().manual_seed(11), "reject")
assert reject_tokens57 == [1]
assert reject_trace57["accepted"] == 0 and reject_trace57["rejected"] == 1
assert reject_trace57["rollbacks"] >= 1
assert reject_decoder57.target_cache.tokens("reject") == reject_decoder57.draft_cache.tokens("reject")

eos57 = deterministic_matrix57(TOKENIZER57["<eos>"])
eos_decoder57 = SpeculativeDecoder57(BigramLM57(eos57), BigramLM57(eos57), gamma=3)
eos_tokens57, eos_trace57 = eos_decoder57([0], 6, torch.Generator().manual_seed(12), "eos")
assert eos_tokens57 == [TOKENIZER57["<eos>"]]
assert eos_trace57["bonuses"] == 0
assert eos_trace57["accepted"] == 1

def add_mass57(table, key, mass):
    table[key] = table.get(key, 0.0) + float(mass)

def enumerate_two_token_spec57(decoder, prefix_token):
    p0 = decoder.target.next_probs(prefix_token)
    q0 = decoder.draft.next_probs(prefix_token)
    residual0 = None if torch.allclose(p0, q0, atol=1e-15) else residual_distribution57(p0, q0)
    outcomes = {}
    for proposal in range(decoder.target.vocab_size):
        accepted_mass = torch.minimum(p0[proposal], q0[proposal])
        if accepted_mass > 0:
            if proposal == decoder.eos_token:
                add_mass57(outcomes, (proposal,), accepted_mass)
            else:
                bonus_p = decoder.target.next_probs(proposal)
                for bonus in range(decoder.target.vocab_size):
                    add_mass57(outcomes, (proposal, bonus), accepted_mass * bonus_p[bonus])
        rejected_mass = (q0[proposal] - p0[proposal]).clamp_min(0)
        if rejected_mass > 0:
            for replacement in range(decoder.target.vocab_size):
                replacement_mass = rejected_mass * residual0[replacement]
                if replacement_mass <= 0:
                    continue
                if replacement == decoder.eos_token:
                    add_mass57(outcomes, (replacement,), replacement_mass)
                else:
                    p1 = decoder.target.next_probs(replacement)
                    q1 = decoder.draft.next_probs(replacement)
                    second = exact_first_output57(p1, q1)
                    for token2 in range(decoder.target.vocab_size):
                        add_mass57(outcomes, (replacement, token2), replacement_mass * second[token2])
    return outcomes

def enumerate_two_token_target57(decoder, prefix_token):
    outcomes = {}
    p0 = decoder.target.next_probs(prefix_token)
    for first in range(decoder.target.vocab_size):
        if first == decoder.eos_token:
            add_mass57(outcomes, (first,), p0[first])
        else:
            p1 = decoder.target.next_probs(first)
            for second in range(decoder.target.vocab_size):
                add_mass57(outcomes, (first, second), p0[first] * p1[second])
    return outcomes

distribution_decoder57 = SpeculativeDecoder57(
    BigramLM57(TARGET_PROBS57), BigramLM57(DRAFT_PROBS57), gamma=1, eos_token=TOKENIZER57["<eos>"]
)
exact_spec_two57 = enumerate_two_token_spec57(distribution_decoder57, TOKENIZER57["<bos>"])
exact_target_two57 = enumerate_two_token_target57(distribution_decoder57, TOKENIZER57["<bos>"])
all_outcomes57 = set(exact_spec_two57) | set(exact_target_two57)
assert abs(sum(exact_spec_two57.values()) - 1.0) < 1e-7
assert abs(sum(exact_target_two57.values()) - 1.0) < 1e-7
assert max(abs(exact_spec_two57.get(key, 0.0) - exact_target_two57.get(key, 0.0)) for key in all_outcomes57) < 2e-7
assert any(len(key) == 1 and key[0] == TOKENIZER57["<eos>"] for key in exact_spec_two57)
assert any(len(key) == 2 for key in exact_spec_two57)

empirical_counts57 = {}
for seed57 in range(500):
    request_id57 = "dist-" + str(seed57)
    tokens57, _ = distribution_decoder57(
        [TOKENIZER57["<bos>"]], 2, torch.Generator().manual_seed(seed57), request_id57
    )
    distribution_decoder57.release_request(request_id57)
    key57 = tuple(tokens57)
    empirical_counts57[key57] = empirical_counts57.get(key57, 0) + 1
empirical_max_error57 = max(
    abs(empirical_counts57.get(key, 0) / 500 - exact_target_two57.get(key, 0.0))
    for key in all_outcomes57
)
assert empirical_max_error57 < 0.07

## 8. Generator 可复现与 batch/request 位置隔离

随机源必须由调用方显式提供，不能混用 Python random、全局 torch RNG 与设备 RNG。相同模型、prefix 和 seed 应产生相同 token/trace。批请求只是教学版逐请求循环，但每个 request_id 有独立 cache 和独立 generator；不同 prefix 长度下位置仍等于 prefix_len+generated_len。

In [ ]:
deterministic_a57 = SpeculativeDecoder57(BigramLM57(TARGET_PROBS57), BigramLM57(DRAFT_PROBS57), gamma=2)
deterministic_b57 = SpeculativeDecoder57(BigramLM57(TARGET_PROBS57), BigramLM57(DRAFT_PROBS57), gamma=2)
output_a57 = deterministic_a57([0], 4, torch.Generator().manual_seed(99), "same")
output_b57 = deterministic_b57([0], 4, torch.Generator().manual_seed(99), "same")
assert output_a57 == output_b57

batch_decoder57 = SpeculativeDecoder57(BigramLM57(TARGET_PROBS57), BigramLM57(DRAFT_PROBS57), gamma=2)
requests57 = [{"request_id": "batch-a", "prefix": [0]}, {"request_id": "batch-b", "prefix": [0, 1]}]
batch_outputs57 = batch_decoder57.forward_batch(
    requests57, 3, [torch.Generator().manual_seed(101), torch.Generator().manual_seed(202)]
)
assert set(batch_outputs57) == {"batch-a", "batch-b"}
for request57 in requests57:
    request_id57 = request57["request_id"]
    generated_count57 = len(batch_outputs57[request_id57][0])
    assert batch_decoder57.target_cache.position(request_id57) == len(request57["prefix"]) + generated_count57
    assert batch_decoder57.target_cache.tokens(request_id57) == batch_decoder57.draft_cache.tokens(request_id57)
try:
    batch_decoder57([0], 1, torch.Generator().manual_seed(1), "batch-a")
    raise AssertionError("live request id was reused")
except RuntimeError as error57:
    assert str(error57) == "request_already_exists"
batch_decoder57.release_request("batch-a"); batch_decoder57.release_request("batch-b")

## 9. 教学循环与生产并行验证的差异

本例逐 token 调 target，算法分布正确但没有性能优势。生产 speculative decoding 会让 draft 先生成 gamma 个 token，再用 target 一次并行 forward 验证整块；每层 KV cache 需要可提交/回滚的页表，连续批处理还要处理不同请求的接受长度。

收益取决于接受率、gamma、draft 延迟、target 并行效率和内存带宽。tokenizer、采样温度/top-k/top-p、logit processor、EOS 和随机数消费顺序必须完全对齐；只比较 greedy token 相等不足以证明 sampling 分布正确。

## 10. 可信发布：同时绑定 target、draft 与解码 recipe

state 摘要以 target:: 和 draft:: 命名空间绑定每个 key、dtype、shape、bytes。metadata 绑定完整 tokenizer、评估 prompts、split、gamma、EOS、最大输出、采样算法与 cache 语义。

包外 MappingProxy registry 是信任根。loader 返回 PublishedSpeculative57，包装器为每次请求创建显式 generator，并在 finally 中释放 cache，避免异常请求遗留状态污染下一次调用。

In [ ]:
def state_digest57(state):
    digest = hashlib.sha256()
    for key in sorted(state):
        tensor = state[key].detach().cpu().contiguous()
        header = canonical_json57({"key": key, "dtype": str(tensor.dtype), "shape": list(tensor.shape)}).encode()
        raw = tensor.numpy().tobytes()
        digest.update(len(header).to_bytes(8, "big")); digest.update(header)
        digest.update(len(raw).to_bytes(8, "big")); digest.update(raw)
    return digest.hexdigest()

CONFIG57 = {"vocab_size": 4}
RECIPE57 = {
    "algorithm": "speculative_sampling_with_positive_residual",
    "gamma": 2, "eos_token": 3, "max_new_tokens": 8,
    "temperature": 1.0, "top_k": None, "top_p": None,
    "explicit_generator": True, "rollback_unit": "committed_token_position",
}

def flatten_states57(target, draft):
    state = {}
    for prefix, model in (("target", target), ("draft", draft)):
        for key, value in model.state_dict().items():
            state[prefix + "::" + key] = value.detach().cpu().clone()
    return state

def release_digest57(package):
    envelope = {
        "release_id": package["release_id"], "metadata": package["metadata"],
        "state_digest": state_digest57(package["state"]),
    }
    return sha57(canonical_json57(envelope).encode())

def build_release57(target, draft):
    state = flatten_states57(target, draft)
    package = {
        "release_id": "spec-bigram-v1",
        "metadata": {
            "config": copy.deepcopy(CONFIG57), "tokenizer": copy.deepcopy(TOKENIZER57),
            "data": copy.deepcopy(PROMPTS57), "split": copy.deepcopy(SPLIT57),
            "recipe": copy.deepcopy(RECIPE57), "allowed_subject": "inference-lab",
        },
        "state": state, "state_digest": state_digest57(state),
    }
    package["self_digest"] = release_digest57(package)
    return package

RELEASE_PACKAGE57 = build_release57(target_model57, draft_model57)
TRUSTED_RELEASES57 = MappingProxyType({"spec-bigram-v1": release_digest57(RELEASE_PACKAGE57)})

class PublishedSpeculative57(nn.Module):
    def __init__(self, target, draft, metadata, subject):
        super().__init__()
        self.decoder = SpeculativeDecoder57(target, draft, metadata["recipe"]["gamma"], metadata["recipe"]["eos_token"])
        self.tokenizer = MappingProxyType(copy.deepcopy(metadata["tokenizer"]))
        self.subject = subject
        self.max_new_tokens = metadata["recipe"]["max_new_tokens"]

    def forward(self, prefix, max_new_tokens, seed, request_id):
        if self.subject != "inference-lab":
            raise PermissionError("subject_not_authorized")
        if not isinstance(seed, int):
            raise TypeError("seed_must_be_int")
        if max_new_tokens > self.max_new_tokens:
            raise ValueError("request_exceeds_published_limit")
        generator = torch.Generator().manual_seed(seed)
        try:
            return self.decoder(prefix, max_new_tokens, generator, request_id)
        finally:
            if request_id in self.decoder.target_cache._requests:
                self.decoder.release_request(request_id)

def validate_metadata57(metadata):
    tokenizer = metadata["tokenizer"]
    if tokenizer != TOKENIZER57 or sorted(tokenizer.values()) != list(range(len(tokenizer))):
        raise ValueError("tokenizer_contract_mismatch")
    data, split = metadata["data"], metadata["split"]
    ids, referenced = [row["id"] for row in data], sum(split.values(), [])
    if len(ids) != len(set(ids)) or sorted(ids) != sorted(referenced) or len(referenced) != len(set(referenced)):
        raise ValueError("data_split_contract_mismatch")
    if any(not row["tokens"] or any(type(token) is not int or token not in tokenizer.values() for token in row["tokens"]) for row in data):
        raise ValueError("invalid_bound_prompts")
    if metadata["recipe"] != RECIPE57 or metadata["config"]["vocab_size"] != len(tokenizer):
        raise ValueError("decoder_recipe_mismatch")

def load_published57(package, subject):
    release_id = package.get("release_id")
    actual = release_digest57(package)
    if release_id not in TRUSTED_RELEASES57 or actual != TRUSTED_RELEASES57[release_id]:
        raise PermissionError("untrusted_release_digest")
    if package.get("self_digest") != actual or package.get("state_digest") != state_digest57(package["state"]):
        raise ValueError("corrupt_release")
    validate_metadata57(package["metadata"])
    if subject != package["metadata"]["allowed_subject"]:
        raise PermissionError("subject_not_authorized")
    vocab_size = package["metadata"]["config"]["vocab_size"]
    uniform = torch.full((vocab_size, vocab_size), 1.0 / vocab_size, dtype=torch.float64)
    target, draft = BigramLM57(uniform), BigramLM57(uniform)
    target_state = {key.split("::", 1)[1]: value for key, value in package["state"].items() if key.startswith("target::")}
    draft_state = {key.split("::", 1)[1]: value for key, value in package["state"].items() if key.startswith("draft::")}
    target.load_state_dict(target_state, strict=True); draft.load_state_dict(draft_state, strict=True)
    return PublishedSpeculative57(target, draft, package["metadata"], subject)

published57 = load_published57(copy.deepcopy(RELEASE_PACKAGE57), "inference-lab")
published_tokens57, published_trace57 = published57([0], 4, 303, "published-a")
assert 1 <= len(published_tokens57) <= 4
assert published_trace57["blocks"] >= 1
assert published57.decoder.target_cache._requests == {}
assert published57.decoder.draft_cache._requests == {}
assert isinstance(TRUSTED_RELEASES57, MappingProxyType)

## 11. 整体重签、非法请求与失败恢复

下列攻击同时修改 target state、更新 state_digest 和 self_digest，包内字段完全一致；包外 registry 仍拒绝。包装器还拒绝超出发布上限的输出，并确认即使 forward 抛错，finally 也不会遗留请求 cache。

In [ ]:
forged57 = copy.deepcopy(RELEASE_PACKAGE57)
forged_key57 = next(key for key in forged57["state"] if key.startswith("target::"))
forged57["state"][forged_key57].view(-1)[0] += 0.25
forged57["state_digest"] = state_digest57(forged57["state"])
forged57["self_digest"] = release_digest57(forged57)
assert forged57["self_digest"] == release_digest57(forged57)
try:
    load_published57(forged57, "inference-lab")
    raise AssertionError("fully resigned forged decoder was trusted")
except PermissionError as error57:
    assert str(error57) == "untrusted_release_digest"

try:
    load_published57(RELEASE_PACKAGE57, "outsider")
    raise AssertionError("unauthorized subject was accepted")
except PermissionError as error57:
    assert str(error57) == "subject_not_authorized"
try:
    published57([0], 9, 1, "too-long")
    raise AssertionError("request above published limit was accepted")
except ValueError as error57:
    assert str(error57) == "request_exceeds_published_limit"
assert published57.decoder.target_cache._requests == {}
try:
    published57([3], 2, 1, "ended")
    raise AssertionError("already-ended prefix was accepted")
except ValueError as error57:
    assert str(error57) == "prefix_already_ended"
assert published57.decoder.draft_cache._requests == {}

## 12. 结论、复杂度与原始资料

本例先精确穷举每个上下文的首输出，再通过实际 decoder 的两 token 路径解析枚举与 500 次采样验证 EOS、拒绝续块和 bonus 的联合分布等于 target，并覆盖 p/q、q=0、全接受、全拒绝、bonus、EOS、显式 generator、批请求位置和 rollback。算法减少的是昂贵 target 的串行调用次数，不改变 target 分布；如果 residual、随机数或 cache 提交任一处写错，就可能只在采样模式下悄悄偏分布。

原始资料：

- Fast Inference from Transformers via Speculative Decoding：https://arxiv.org/abs/2211.17192
- Accelerating Large Language Model Decoding with Speculative Sampling：https://arxiv.org/abs/2302.01318
- PyTorch multinomial 官方文档：https://pytorch.org/docs/stable/generated/torch.multinomial.html